In [ ]:
#import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_excel("C:\\Users\\dell\\Machine Learning Assignment\\26-Apr\\dataset.xlsx")
data

In [ ]:
missing_values = data.isna().sum()/data.shape[0] #sum() to count the number of missing values in each column
#print(data.isnull().mean())                   #mean() to calculate the fraction of missing values in each column

# set the display options to show all columns
pd.set_option('display.max_columns', None) #Displays all columns in a dataframe, without truncation
pd.set_option('display.max_rows', None)    #Displays all rows in a dataframe, without truncation
pd.set_option('display.width', None)       #Displays the dataframe without line wrapping
pd.set_option('display.max_colwidth', -1)  #Displays all the text in each cell, without truncation

print(missing_values)

In [ ]:
#Selects the columns with missing data between 86% and 90% and assigns them to a list called "blood"

blood = list(data.columns[(missing_values > 0.86) & (missing_values < 0.9)])
print(blood)

In [ ]:
#Selecting columns with missing values greater than 70% and less than 80%

virus = list(data.columns[(missing_values > 0.7) & (missing_values < 0.8)])
print(virus)

In [ ]:
data = data[['SARS-Cov-2 exam result'] + blood + virus]

In [ ]:
# Encode categorical variables

def encode(data): 
    encoder = {"not_detected": 0, "negative": 0, "detected": 1, "positive": 1}
    for column in data.select_dtypes(include=['object']):
        data.loc[:, column] = data[column].map(encoder)
    return data

In [ ]:
# Impute missing values

def impute(data):
    data = data.fillna(data.median())
    return data

In [ ]:
# Chain the two functions together
data = encode(data)
data = impute(data)


In [ ]:
X = data[blood + virus]
y = data['SARS-Cov-2 exam result']

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)


In [ ]:
# Apply SMOTE to balance class distribution

from imblearn.over_sampling import SMOTE

# Concatenate training and test sets
#X = np.concatenate([X_train_pca, X_test_pca], axis=0)
#y = np.concatenate([y_train, y_test], axis=0)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# Print class distribution before and after SMOTE
print('Class distribution before SMOTE:', y_train.value_counts())
print('Class distribution after SMOTE:', y_resampled.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, precision_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

# Resample test set using SMOTE
X_train_resampled, X_test_resampled, y_train_resampled, y_test_resampled = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)

# Train and evaluate models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(DecisionTreeClassifier(random_state=42), random_state=42, learning_rate=0.1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_resampled)
    acc = accuracy_score(y_test_resampled, y_pred)
    cm = confusion_matrix(y_test_resampled, y_pred)
    f1 = f1_score(y_test_resampled, y_pred, average='weighted')
    recall = recall_score(y_test_resampled, y_pred, average='weighted')
    precision = precision_score(y_test_resampled, y_pred, zero_division=0, average='weighted')
    roc_auc = roc_auc_score(y_test_resampled, y_pred)
    fpr, tpr, thresholds = roc_curve(y_test_resampled, y_pred)
    print(name)
    print(f"Accuracy: {acc:.4f}")
    print(f"Confusion matrix:\n{cm}")
    print(f"F1 score: {f1:.4f}")
    print(f"Recall score: {recall:.4f}")
    print(f"Precision score: {precision:.4f}")
    print(f"AUC score: {roc_auc:.4f}")
    print("--------------")    
    
    # Plot confusion matrix
    ConfusionMatrixDisplay.from_estimator(model, X_test_resampled, y_test_resampled, cmap=plt.cm.Blues)
    plt.title(f"{name} Confusion Matrix")
    plt.show()
    
    # Plot ROC curve
    roc_display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name=name)
    roc_display.plot()
    plt.title(f"{name} ROC Curve")
    plt.show()


In [ ]:

# Train and evaluate models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(DecisionTreeClassifier(random_state=42), random_state=42, learning_rate=0.1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Create empty lists to store results
model_names = []
accuracies = []
f1_scores = []
recalls = []
precisions = []

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_resampled)
    acc = accuracy_score(y_test_resampled, y_pred)
    cm = confusion_matrix(y_test_resampled, y_pred)
    f1 = f1_score(y_test_resampled, y_pred, average='weighted')
    recall = recall_score(y_test_resampled, y_pred, average='weighted')
    precision = precision_score(y_test_resampled, y_pred, zero_division=0, average='weighted')
    roc_auc = roc_auc_score(y_test_resampled, y_pred)
    fpr, tpr, thresholds = roc_curve(y_test_resampled, y_pred)
    print(name)
    print(f"Accuracy: {acc:.4f}")
    print(f"Confusion matrix:\n{cm}")
    print(f"F1 score: {f1:.4f}")
    print(f"Recall score: {recall:.4f}")
    print(f"Precision score: {precision:.4f}")
    print(f"AUC score: {roc_auc:.4f}")
    print("--------------")    
    
    # Add results to lists
    model_names.append(name)
    accuracies.append(acc)
    f1_scores.append(f1)
    recalls.append(recall)
    precisions.append(precision)

# Create dataframe of results
results_df = pd.DataFrame({
    'Model': model_names,
    'Accuracy': accuracies,
    'F1 Score': f1_scores,
    'Recall': recalls,
    'Precision': precisions
})

# Print results table
print(results_df)


In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Instantiate the random forest classifier
rfc = RandomForestClassifier(random_state=42)

# Instantiate the grid search
grid_search = GridSearchCV(rfc, param_grid, cv=5, scoring='accuracy')

# Fit the grid search to the resampled training data
grid_search.fit(X_train_resampled, y_train_resampled)

# Print the best hyperparameters and the corresponding mean test score
print("Best Hyperparameters:", grid_search.best_params_)
print("Best Mean Test Score:", grid_search.best_score_)


In [ ]:
# Instantiate the random forest classifier with the best hyperparameters
rfc_final = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, min_samples_leaf=1, random_state=42)

# Fit the model to the resampled training data
rfc_final.fit(X_train_resampled, y_train_resampled)

# Predict the target variable on the test set
y_pred = rfc_final.predict(X_test)

# Print the accuracy score on the test set
print("Accuracy Score:", accuracy_score(y_test, y_pred))
